# Day 29 — **pytest: Mocking LLM Calls**

**Phase 1 · Module 4: Agentic System Design Patterns** · ~30 min

You can't unit-test an agent that makes a real Bedrock call on every run — it's slow, costs money, and returns different text each time. The fix is **mocking**: `unittest.mock.patch` swaps the LLM client for a fake you control, so tests are fast, free, deterministic, and offline. This is what makes the testing layers from today's LangGraph lesson actually runnable in CI.

Today:
1. Code that calls an LLM client (the thing to mock).
2. `patch` + `MagicMock` — replace the client, control its return.
3. `side_effect` — sequences and simulated errors.
4. Asserting *how* the client was called (`assert_called_once_with`).

## 1. Code that calls an LLM client

`summarise_case` depends on a module-level `bedrock` client. In production that's a real boto3 client; in a test we'll replace it — without touching this function's code.

In [1]:
class BedrockClient:
    def invoke(self, prompt: str) -> str:
        raise RuntimeError("real network call — must never run in a unit test")

bedrock = BedrockClient()               # module-level dependency

def summarise_case(case_notes: str) -> dict:
    prompt = f"Summarise this fraud case in one line:\n{case_notes}"
    text = bedrock.invoke(prompt)
    return {"summary": text.strip(), "chars": len(case_notes)}

print("function defined; calling it for real would raise:")
try: summarise_case("notes")
except RuntimeError as e: print(" ->", e)

function defined; calling it for real would raise:
 -> real network call — must never run in a unit test


☝ Calling it for real raises — deliberately, to prove the test never hits the network. The dependency is a *module-level name* (`bedrock`), which is exactly what `patch` targets: it temporarily rebinds that name to a fake for the duration of the test, then restores it. Design your code so the client is patchable (a module global or an injected argument), not buried inside the function.

## 2. `patch` + `MagicMock` — control the return

`patch("__main__.bedrock")` replaces the client object. The replacement is a `MagicMock` whose `.invoke(...)` returns whatever we set. The function under test runs its real logic against our canned response.

In [2]:
from unittest.mock import patch, MagicMock

def test_summarise_happy():
    fake = MagicMock()
    fake.invoke.return_value = "  Customer disputed 3 card payments  "
    with patch("__main__.bedrock", fake):
        out = summarise_case("long fraud notes here")
    assert out["summary"] == "Customer disputed 3 card payments"   # .strip() applied
    assert out["chars"] == len("long fraud notes here")
    print("PASS: happy path ->", out)

test_summarise_happy()

PASS: happy path -> {'summary': 'Customer disputed 3 card payments', 'chars': 21}


☝ No network, instant, deterministic. `return_value` makes `fake.invoke(...)` always yield our canned string, so we can assert that `summarise_case` correctly `.strip()`s it and counts the input. `patch` as a context manager auto-restores the real `bedrock` on exit — later tests aren't polluted. This is the backbone of every fast agent test suite.

## 3. `side_effect` — sequences and errors

`return_value` is one fixed answer. `side_effect` is richer: a **list** yields a different value per call (for multi-turn agents), and an **exception** class simulates a failure so you can test error handling.

In [3]:
def test_side_effect_sequence():
    fake = MagicMock()
    fake.invoke.side_effect = ["first reply", "second reply"]
    with patch("__main__.bedrock", fake):
        a = summarise_case("case A")["summary"]
        b = summarise_case("case B")["summary"]
    assert (a, b) == ("first reply", "second reply")
    print("PASS: sequence ->", a, "|", b)

def test_error_handling():
    fake = MagicMock()
    fake.invoke.side_effect = TimeoutError("bedrock timed out")
    with patch("__main__.bedrock", fake):
        try:
            summarise_case("case C"); assert False, "should propagate"
        except TimeoutError as e:
            print("PASS: error surfaced ->", e)

test_side_effect_sequence()
test_error_handling()

PASS: sequence -> first reply | second reply
PASS: error surfaced -> bedrock timed out


☝ `side_effect=[...]` returns each item on successive calls — essential for testing a ReAct loop that queries the model several times. `side_effect=SomeError` *raises* instead, letting you prove your retry/timeout handling works **without waiting for a real timeout**. Testing the sad path is where mocking earns its keep: real errors are hard to trigger on demand; mocked ones are trivial.

## 4. Assert *how* it was called

Sometimes the bug isn't the return value — it's that you sent the model the wrong prompt. Mock objects record every call, so you can assert the exact arguments with `assert_called_once_with` and inspect `call_args`.

In [4]:
def test_prompt_construction():
    fake = MagicMock()
    fake.invoke.return_value = "ok"
    with patch("__main__.bedrock", fake):
        summarise_case("dispute on AC-1001")
    # was invoke called exactly once, with a prompt containing the notes?
    fake.invoke.assert_called_once()
    sent_prompt = fake.invoke.call_args.args[0]
    assert "dispute on AC-1001" in sent_prompt
    assert sent_prompt.startswith("Summarise this fraud case")
    print("PASS: prompt built correctly ->", repr(sent_prompt[:40]), "...")

test_prompt_construction()

PASS: prompt built correctly -> 'Summarise this fraud case in one line:\nd' ...


☝ The mock is also a **spy**: `assert_called_once()` catches accidental double-calls (wasted tokens), and `call_args` lets you verify the *prompt itself* — the most common place agent bugs hide. A test that only checks the return value would miss a prompt that forgot to include the case notes. Assert on both the inputs you send and the outputs you get.

## 5. Recap — mock the model, test everything else

| Tool | Use |
|---|---|
| `patch("mod.name", fake)` | swap the LLM client for a fake (context manager auto-restores) |
| `MagicMock().return_value` | one canned response |
| `side_effect=[...]` | different response per call (multi-turn agents) |
| `side_effect=SomeError` | simulate timeouts/failures for error-path tests |
| `assert_called_once_with` / `call_args` | verify the prompt you sent |

Mocking draws a clean line: the LLM's *content* is the model provider's problem; your *logic around it* is yours, and it must be tested fast and offline. Patch the client, and the unit/contract/mutation tests from today's LangGraph lesson all run in milliseconds on every commit — no key, no cost, no flakiness.